<a href="https://colab.research.google.com/github/0xdaxl/soc-ml-recommendation-engine/blob/main/notebooks/SOC_ML_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-generativeai

In [3]:

import google.generativeai as genai
import json

# ============================================================
# CONFIGURATION
# ============================================================
from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

# ============================================================
# SIMULATED WAZUH ALERTS (from 03-Dataset.md)
# ============================================================

brute_force_alert = {
    "timestamp": "2026-03-19T03:42:17.000Z",
    "rule": {
        "id": "5763",
        "description": "SSHD brute force trying to get access to the system",
        "level": 10,
        "groups": ["authentication_failures", "sshd", "brute_force"]
    },
    "agent": {"id": "003", "name": "healthcare-workstation-03", "ip": "10.10.20.15"},
    "data": {
        "srcip": "192.168.1.45",
        "dstuser": "admin",
        "failed_attempts": 47,
        "protocol": "ssh",
        "port": "22"
    }
}

malware_alert = {
    "timestamp": "2026-03-19T11:15:33.000Z",
    "rule": {
        "id": "87105",
        "description": "Malware detected — suspicious executable found",
        "level": 12,
        "groups": ["malware", "rootcheck", "syscheck"]
    },
    "agent": {"id": "007", "name": "healthcare-server-ehr", "ip": "10.10.20.100"},
    "data": {
        "file": "/tmp/.hidden/svchost32.exe",
        "md5": "d41d8cd98f00b204e9800998ecf8427e",
        "process": "svchost32.exe",
        "user": "ehr-service",
        "action": "created"
    }
}

privesc_alert = {
    "timestamp": "2026-03-19T22:07:55.000Z",
    "rule": {
        "id": "5402",
        "description": "Successful sudo to root executed",
        "level": 9,
        "groups": ["authentication_success", "sudo", "privilege_escalation"]
    },
    "agent": {"id": "005", "name": "healthcare-linux-workstation", "ip": "10.10.20.51"},
    "data": {
        "srcuser": "nurse_account_12",
        "dstuser": "root",
        "command": "sudo su -",
        "program_name": "sudo"
    }
}

# ============================================================
# COMPLIANCE RULES (embedded per alert type)
# ============================================================

compliance_map = {
    "brute_force": """
HIPAA §164.312(d) — Person/Entity Authentication:
Implement procedures to verify that a person or entity seeking access
to ePHI is who they claim to be. Failed authentication must be monitored.

HIPAA §164.308(a)(5)(ii)(C) — Log-in Monitoring:
Procedures for monitoring log-in attempts and reporting discrepancies.

NIST AC-7 — Unsuccessful Logon Attempts:
Enforce a limit on consecutive invalid logon attempts. Lock the account
after the threshold is exceeded.

NIST IA-5 — Authenticator Management:
Manage system authenticators including passwords, tokens, and certificates.
    """,

    "malware": """
HIPAA §164.308(a)(1)(ii)(A) — Risk Analysis:
Conduct an accurate and thorough assessment of potential risks to ePHI.

HIPAA §164.308(a)(5)(ii)(B) — Protection from Malicious Software:
Procedures for guarding against, detecting, and reporting malicious software.

NIST SI-3 — Malicious Code Protection:
Implement malicious code protection at system entry/exit points.
Take action when malicious code is detected including alerting administrators.

NIST IR-4 — Incident Handling:
Implement an incident handling capability including preparation,
detection, analysis, containment, eradication, and recovery.
    """,

    "privilege_escalation": """
HIPAA §164.312(a)(1) — Access Control — Unique User Identification:
Assign a unique name/number for identifying and tracking user identity.
Clinician accounts must only have access required for their role.

HIPAA §164.308(a)(3) — Workforce Access Management:
Implement policies to authorize and supervise workforce access to ePHI.

NIST AC-6 — Least Privilege:
Employ the principle of least privilege, allowing only authorized access
required for users to accomplish assigned tasks.

NIST AC-2 — Account Management:
Manage system accounts including establishing, activating, and monitoring.
Immediately disable accounts when no longer required or authorized.
    """
}

# ============================================================
# ALERT TYPE DETECTION
# ============================================================

def detect_alert_type(alert):
    groups = alert.get("rule", {}).get("groups", [])
    if "brute_force" in groups or "authentication_failures" in groups:
        return "brute_force"
    elif "malware" in groups or "syscheck" in groups:
        return "malware"
    elif "privilege_escalation" in groups or "sudo" in groups:
        return "privilege_escalation"
    else:
        return "brute_force"  # default fallback

# ============================================================
# PROMPT BUILDER
# ============================================================

def build_prompt(alert, alert_type):
    compliance = compliance_map[alert_type]
    alert_str = json.dumps(alert, indent=2)

    prompt = f"""You are an expert SOC analyst specializing in healthcare cybersecurity.
You have deep knowledge of HIPAA Security Rule and NIST SP 800-53 controls.
You are analyzing alerts from a Wazuh SIEM deployed in a healthcare environment
that handles Electronic Health Records (EHR) and Protected Health Information (PHI).
Your job is to give clear, actionable recommendations to SOC analysts.

RELEVANT COMPLIANCE RULES FOR THIS ALERT:
{compliance}

WAZUH ALERT TO ANALYZE:
{alert_str}

Provide your analysis in EXACTLY this format — do not deviate:

**WHAT HAPPENED:** (2-3 sentences explaining the event in plain language for a SOC analyst)

**THREAT LEVEL:** (Choose one: CRITICAL / HIGH / MEDIUM / LOW — then one sentence justification)

**COMPLIANCE VIOLATION:** (Which specific HIPAA section and NIST control is violated, and how)

**IMMEDIATE ACTIONS:** (Numbered list — what the analyst must do in the next 15 minutes)

**INVESTIGATION STEPS:** (Numbered list — what to check next to understand the full scope)

**CASE NOTES FOR THEHIVE:** (A short paragraph to paste directly into the incident case)
"""
    return prompt

# ============================================================
# RECOMMENDATION ENGINE — MAIN FUNCTION
# ============================================================

def get_recommendation(alert):
    alert_type = detect_alert_type(alert)
    prompt = build_prompt(alert, alert_type)

    print(f"\n{'='*60}")
    print(f"ALERT TYPE DETECTED: {alert_type.upper().replace('_', ' ')}")
    print(f"AGENT: {alert['agent']['name']} ({alert['agent']['ip']})")
    print(f"RULE: {alert['rule']['description']}")
    print(f"SEVERITY LEVEL: {alert['rule']['level']}/15")
    print(f"{'='*60}\n")

    response = model.generate_content(prompt)

    print("GEMINI SOC RECOMMENDATION:")
    print("-" * 60)
    print(response.text)
    print("-" * 60)

    return response.text

# ============================================================
# RUN ALL 3 USE CASES
# ============================================================

print("RUNNING USE CASE 1: BRUTE FORCE")
get_recommendation(brute_force_alert)

print("\n\nRUNNING USE CASE 2: MALWARE DETECTION")
get_recommendation(malware_alert)

print("\n\nRUNNING USE CASE 3: PRIVILEGE ESCALATION")
get_recommendation(privesc_alert)

RUNNING USE CASE 1: BRUTE FORCE

ALERT TYPE DETECTED: BRUTE FORCE
AGENT: healthcare-workstation-03 (10.10.20.15)
RULE: SSHD brute force trying to get access to the system
SEVERITY LEVEL: 10/15

GEMINI SOC RECOMMENDATION:
------------------------------------------------------------
**WHAT HAPPENED:**
A persistent brute-force attack via SSH is targeting the 'admin' account on `healthcare-workstation-03` (IP 10.10.20.15) from an internal source IP (192.168.1.45). The attacker made 47 unsuccessful login attempts against this system, which handles ePHI, without the account being locked out.

**THREAT LEVEL:**
CRITICAL – This is an active internal brute-force attack targeting a privileged account on a system handling PHI, indicating a potential insider threat or a compromised internal host.

**COMPLIANCE VIOLATION:**
*   **HIPAA §164.312(d) — Person/Entity Authentication:** The failure to automatically lock the 'admin' account after 47 unsuccessful attempts demonstrates a lack of effective v

"**WHAT HAPPENED:** A non-administrative user account, identified as `nurse_account_12`, successfully executed `sudo su -` to gain full root privileges on the `healthcare-linux-workstation` (10.10.20.51). This indicates a critical breakdown in access control, where an account designed for a specific clinical role was able to elevate its permissions to the highest level, providing complete control over the system.\n\n**THREAT LEVEL:** CRITICAL — Unauthorized root access by a regular user account allows complete system control, posing an immediate and severe risk to data integrity, confidentiality, and system availability, especially on a workstation handling EHR/PHI.\n\n**COMPLIANCE VIOLATION:**\n*   **HIPAA §164.312(a)(1) — Access Control — Unique User Identification:** Violated as the system failed to ensure that a clinician account (`nurse_account_12`) only had access required for its role, allowing it to gain unauthorized administrative privileges.\n*   **HIPAA §164.308(a)(3) — Work